In [1]:
# ==========================================================
# PV POWER MODEL TRAINING
# ANN + StandardScaler
# No Lag Feature
# ==========================================================

import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
import os

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
tf.random.set_seed(42)

# ==========================================================
# LOAD DATA
# ==========================================================

file_path = r"E:\MTP_Data\MTP_FINAL_READY.csv"

df = pd.read_csv(file_path)
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.set_index("Timestamp")

# ==========================================================
# FEATURES (NO LAG)
# ==========================================================

FEATURE_COLUMNS = [
    "Irradiation_Avg",
    "Environment_Temperature_Avg",
    "WindSpeed_mps",
    "Humidity_%"
]

target = "Total_Power_kW"

X = df[FEATURE_COLUMNS]
y = df[target].values

# ==========================================================
# STANDARD SCALING
# ==========================================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ==========================================================
# BUILD ANN MODEL
# ==========================================================

ann_model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu',
                          input_shape=(X_scaled.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)
])

ann_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.0005),
    loss='mse'
)

# ==========================================================
# TRAIN ANN
# ==========================================================

ann_model.fit(
    X_scaled, y,
    validation_split=0.1,
    epochs=300,
    batch_size=32,
    verbose=0
)

# ==========================================================
# PERFORMANCE
# ==========================================================

y_pred = ann_model.predict(X_scaled, verbose=0).flatten()

rmse = np.sqrt(mean_squared_error(y, y_pred))
mae  = mean_absolute_error(y, y_pred)
mbe  = np.mean(y_pred - y)
r2   = r2_score(y, y_pred)

mean_measured = np.mean(y)
nrmse = (rmse / mean_measured) * 100
nmbe  = (mbe  / mean_measured) * 100

print("\n================ ANN TRAINING PERFORMANCE ================")
print(f"RMSE:  {rmse:.4f}")
print(f"MAE:   {mae:.4f}")
print(f"MBE:   {mbe:.4f}")
print(f"R²:    {r2:.4f}")
print(f"nRMSE: {nrmse:.2f} %")
print(f"nMBE:  {nmbe:.2f} %")

# ==========================================================
# SAVE MODEL + SCALER
# ==========================================================

os.makedirs("models", exist_ok=True)

ann_model.save("models/ann_standard_model.keras")
joblib.dump(scaler, "models/standard_scaler.pkl")
joblib.dump(FEATURE_COLUMNS, "models/ann_features.pkl")

print("\nModel and scaler saved successfully.")

C:\Users\Cordial Space\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



================ ANN TRAINING PERFORMANCE ================
RMSE:  111.5088
MAE:   38.7033
MBE:   -11.3626
R²:    0.9968
nRMSE: 7.53 %
nMBE:  -0.77 %

Model and scaler saved successfully.


In [7]:
# ==========================================================
# SAVE MODEL + SCALER
# ==========================================================

os.makedirs("models", exist_ok=True)

ann_model.save("models/ann_standard_model.keras")
joblib.dump(scaler, "models/standard_scaler.pkl")
joblib.dump(FEATURE_COLUMNS, "models/ann_features.pkl")

print("\nModel and scaler saved successfully.")


Model and scaler saved successfully.


In [8]:
import tensorflow as tf
import joblib
import pandas as pd

# ==========================================================
# LOAD MODEL + SCALER
# ==========================================================

ann_model = tf.keras.models.load_model("models/ann_standard_model.keras")
scaler = joblib.load("models/standard_scaler.pkl")
FEATURE_COLUMNS = joblib.load("models/ann_features.pkl")

print("Model loaded successfully.")

Model loaded successfully.


In [6]:
# ==========================================================
# FULL OPERATIONAL PV FORECAST SYSTEM
# ANN + StandardScaler
# 10-Minute Resolution
# ==========================================================

import requests
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
import os

# ==========================================================
# CONFIGURATION
# ==========================================================

LAT = -2.0261
LON = 30.3772

BASE_DIR = os.getcwd()
MODEL_DIR = os.path.join(BASE_DIR, "models")

FEATURE_COLUMNS = joblib.load(os.path.join(MODEL_DIR, "ann_features.pkl"))

print("Loading ANN model and scaler...")
ann_model = tf.keras.models.load_model(
    os.path.join(MODEL_DIR, "ann_standard_model.keras")
)

scaler = joblib.load(os.path.join(MODEL_DIR, "standard_scaler.pkl"))

# ==========================================================
# DOWNLOAD WEATHER FORECAST
# ==========================================================

url = (
    f"https://api.open-meteo.com/v1/forecast?"
    f"latitude={LAT}&longitude={LON}"
    f"&hourly=shortwave_radiation,temperature_2m,"
    f"relative_humidity_2m,windspeed_10m"
    f"&forecast_days=7"
    f"&timezone=Africa/Kigali"
)

print("Downloading weather forecast...")
data = requests.get(url).json()

df = pd.DataFrame({
    "Time": data["hourly"]["time"],
    "Irradiation_Avg": data["hourly"]["shortwave_radiation"],
    "Environment_Temperature_Avg": data["hourly"]["temperature_2m"],
    "Humidity_%": data["hourly"]["relative_humidity_2m"],
    "WindSpeed_mps": data["hourly"]["windspeed_10m"]
})

df["Time"] = pd.to_datetime(df["Time"])
df = df.set_index("Time")

# Convert hourly → 10min
df_10min = df.resample("10min").interpolate(method="linear")

# ==========================================================
# DATE SELECTION
# ==========================================================

print("\nSelect Forecast Mode:")
print("1 - Single Date")
print("2 - Date Range")

mode = input("Enter option (1 or 2): ")

available_days = df_10min.index.normalize().unique()

if mode == "1":

    selected_date = pd.to_datetime(
        input("\nEnter date (YYYY-MM-DD): ")
    )

    df_selected = df_10min[
        df_10min.index.normalize() == selected_date
    ].copy()

elif mode == "2":

    start_date = pd.to_datetime(
        input("\nEnter start date (YYYY-MM-DD): ")
    )
    end_date = pd.to_datetime(
        input("Enter end date (YYYY-MM-DD): ")
    )

    if (end_date - start_date).days > 7:
        raise ValueError("Maximum forecast range is 7 days.")

    df_selected = df_10min[
        (df_10min.index.normalize() >= start_date) &
        (df_10min.index.normalize() <= end_date)
    ].copy()

else:
    raise ValueError("Invalid selection.")

# ==========================================================
# ANN FORECASTING
# ==========================================================

print("\nRunning ANN forecast...")

X_future = df_selected[FEATURE_COLUMNS]
X_future_scaled = scaler.transform(X_future)

pred_ann = ann_model.predict(X_future_scaled, verbose=0).flatten()
pred_ann = np.maximum(pred_ann, 0)

forecast_df = df_selected.copy()
forecast_df["ANN_Power_kW"] = pred_ann

# ==========================================================
# ENERGY CALCULATION
# ==========================================================

delta_t = 10 / 60

forecast_df["Energy_kWh"] = forecast_df["ANN_Power_kW"] * delta_t

# Hourly
hourly_energy = forecast_df["Energy_kWh"].resample("h").sum()
hourly_peak   = forecast_df["ANN_Power_kW"].resample("h").max()

# Daily
daily_energy = forecast_df["Energy_kWh"].resample("D").sum()
daily_peak   = forecast_df["ANN_Power_kW"].resample("D").max()

print("\n================ HOURLY ENERGY =================")
print(hourly_energy)

print("\n================ DAILY SUMMARY =================")
for date in daily_energy.index:
    print("\nDate:", date.date())
    print("Total Energy (kWh):", round(daily_energy.loc[date], 2))
    print("Peak Power (kW):", round(daily_peak.loc[date], 2))

print("\n================================================")

Loading ANN model and scaler...

Select Forecast Mode:
1 - Single Date
2 - Date Range

Running ANN forecast...

================ HOURLY ENERGY =================
Time
2026-03-03 00:00:00    0.0
2026-03-03 01:00:00    0.0
2026-03-03 02:00:00    0.0
2026-03-03 03:00:00    0.0
2026-03-03 04:00:00    0.0
                      ... 
2026-03-08 19:00:00    0.0
2026-03-08 20:00:00    0.0
2026-03-08 21:00:00    0.0
2026-03-08 22:00:00    0.0
2026-03-08 23:00:00    0.0
Freq: h, Name: Energy_kWh, Length: 144, dtype: float32

================ DAILY SUMMARY =================

Date: 2026-03-03
Total Energy (kWh): 34324.9
Peak Power (kW): 4752.59

Date: 2026-03-04
Total Energy (kWh): 30142.4
Peak Power (kW): 5698.25

Date: 2026-03-05
Total Energy (kWh): 23146.17
Peak Power (kW): 3343.27

Date: 2026-03-06
Total Energy (kWh): 31070.6
Peak Power (kW): 5221.51

Date: 2026-03-07
Total Energy (kWh): 28626.78
Peak Power (kW): 3974.14

Date: 2026-03-08
Total Energy (kWh): 40795.34
Peak Power (kW): 5745.44

